# Used Car Price Prediction

## 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

## 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [23]:
df = pd.read_csv("cardekho_imputated.csv", index_col=[0])

In [24]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Data Cleaning
### Handling Missing values

* Handling Missing values 
* Handling Duplicates
* Check data type
* Understand the dataset

In [25]:
## Check Null Values
##Check features with nan value
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [26]:
## Remove Unnecessary Columns
df.drop('car_name', axis=1, inplace=True)
df.drop('brand', axis=1, inplace=True)

In [27]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [28]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [29]:
## Indpendent and dependent features
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

In [30]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


## Feature Encoding and Scaling
**One Hot Encoding for Columns which had lesser unique values and not ordinal**
* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [31]:
df.shape

(15411, 11)

In [32]:
df['model'].value_counts()

model
i20            906
Swift Dzire    890
Swift          781
Alto           778
City           757
              ... 
Ghibli           1
Altroz           1
GTC4Lusso        1
Aura             1
Gurkha           1
Name: count, Length: 120, dtype: int64

In [33]:
# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
onehot_columns = ['seller_type','fuel_type','transmission_type']
ordinal_column= ['model']

from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')
ordinal_transformer = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, onehot_columns),
        ("StandardScaler", numeric_transformer, num_features),
        ('Ordinal', ordinal_transformer, ordinal_column )
        
    ],remainder='passthrough')



In [34]:
X=preprocessor.fit_transform(X)

In [35]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022,7.0
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022,54.0
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022,118.0
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022,7.0
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022,38.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022,117.0
15407,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444,42.0
15408,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022,77.0
15409,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444,114.0


In [36]:
# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((12328, 14), (3083, 14))

In [37]:
X_train

array([[ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00, ...,
         2.66249771e+00, -4.03022414e-01,  1.08000000e+02],
       [ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
        -3.86028440e-01, -4.03022414e-01,  9.10000000e+01],
       [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00, ...,
         3.27453006e+00, -4.03022414e-01,  1.70000000e+01],
       ...,
       [ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
        -7.80707856e-01, -4.03022414e-01,  1.00000000e+02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
        -4.35828791e-01, -4.03022414e-01,  1.18000000e+02],
       [ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         6.19420058e-02, -4.03022414e-01,  2.40000000e+01]],
      shape=(12328, 14))

## Model Training And Model Selection

In [38]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [39]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [52]:
## Beginning Model Training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "Adaboost Regressor":AdaBoostRegressor(),
    "Graident BoostRegressor":GradientBoostingRegressor(),
    "Xgboost Regressor":XGBRegressor()
   
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 553855.6665
- Mean Absolute Error: 268101.6071
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502543.5930
- Mean Absolute Error: 279618.5794
- R2 Score: 0.6645


Lasso
Model performance for Training set
- Root Mean Squared Error: 553855.6710
- Mean Absolute Error: 268099.3192
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502542.7117
- Mean Absolute Error: 279614.8406
- R2 Score: 0.6645


Ridge
Model performance for Training set
- Root Mean Squared Error: 553856.3159
- Mean Absolute Error: 268059.9525
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502533.8887
- Mean Absolute Error: 279557.3647
- R2 Score: 0.6645


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 305996.3398
- Mean 

In [41]:
#Initialize few parameter for Hyperparamter tuning

rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

xgboost_params = {"learning_rate": [0.1, 0.01],
                  "max_depth": [5, 8, 12, 20, 30],
                  "n_estimators": [100, 200, 300],
                  "colsample_bytree": [0.5, 0.8, 1, 0.3, 0.4]}

In [42]:
# Models list for Hyperparameter tuning
randomcv_models = [
                   ("RF", RandomForestRegressor(), rf_params),
                   ("XGboost",XGBRegressor(),xgboost_params)
                   
                   ]

In [43]:
##Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
---------------- Best Params for RF -------------------
{'n_estimators': 1000, 'min_samples_split': 2, 'max_features': 5, 'max_depth': 15}
---------------- Best Params for XGboost -------------------
{'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.5}


In [44]:
## Retraining the models with best parameters
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, min_samples_split=2, max_features=5, max_depth=None, 
                                                     n_jobs=-1),
     "Xgboost Regressor":XGBRegressor(n_estimators= 300,learning_rate=0.1,
                                     max_depth=5,colsample_bytree=0.5)}


for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')


Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 131783.7916
- Mean Absolute Error: 39190.1470
- R2 Score: 0.9786
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 208883.4465
- Mean Absolute Error: 97520.4179
- R2 Score: 0.9420


Xgboost Regressor
Model performance for Training set
- Root Mean Squared Error: 111860.5532
- Mean Absolute Error: 73800.4688
- R2 Score: 0.9846
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 300625.3655
- Mean Absolute Error: 103961.8984
- R2 Score: 0.8799




# MODEL PICKLING

Model Pickling simply means saving a Python object to a file so you can load it later without recreating or retraining it.

Think of it like saving a Word document.

- You write a document.
- You save it.
- Later, you open it instead of writing it all over again.

## There are two libaries use for pickling: *Pickle* and *joblib*

| `pickle`                               | `joblib`                                 |
| -------------------------------------- | ---------------------------------------- |
| Built into Python                      | Separate library (`pip install joblib`)  |
| Saves almost any Python object         | Optimized for machine learning objects   |
| Can be slower for large NumPy arrays   | Faster for large NumPy arrays            |
| Produces larger files in many ML cases | Often produces smaller, compressed files |
| General-purpose serialization          | Preferred for scikit-learn models        |


In [45]:
# Train the model
rf_model = RandomForestRegressor(
    n_estimators=200,
    min_samples_split=2,
    max_features=5,
    max_depth=None,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## JOBLIB

In [46]:
from joblib import dump

# Save the trained model
dump(rf_model, "random_forest_model.pkl")

# Save the fitted preprocessor
dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

#### During training: 
The model did not learn from the original data. It learned from the scaled data.

In [47]:
from joblib import load

rf_model = load("random_forest_model.pkl")
preprocessor = load("preprocessor.pkl")

## PICKLE

In [48]:
import pickle

# Save the trained Random Forest model
with open("random_forest_model.pkl", "wb") as f:        #wb:write binary mode
    pickle.dump(rf_model, f)

# Save the fitted preprocessor
with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

In [49]:
# Load the model
with open("random_forest_model.pkl", "rb") as f:        #rb:read binary mode
    rf_model = pickle.load(f)

# Load the preprocessor
with open("preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

In [50]:
sample = pd.DataFrame([{
    "model": "Alto",
    "vehicle_age": 9,
    "km_driven": 120000,
    "seller_type": "Individual",
    "fuel_type": "Petrol",
    "transmission_type": "Manual",
    "mileage": 19.70,
    "engine": 796,
    "max_power": 46.30,
    "seats": 5
}])

In [51]:
import pickle

preprocessor = pickle.load(open("preprocessor.pkl", "rb"))
rf_model = pickle.load(open("random_forest_model.pkl", "rb"))

sample_processed = preprocessor.transform(sample)
predicted_price = rf_model.predict(sample_processed)

print(f"Predicted selling price: {predicted_price[0]:,.2f}")

Predicted selling price: 175,337.50
